# _common_helpers

Shared utilities included by every pipeline notebook via `%run ./_common_helpers`.

**Provides:**
- `log_execution()` — append a row to `metadata.execution_log`
- `log_dqx()` — append a row to `metadata.dqx_execution_log`
- `rollback_table()` — safe Delta time-travel rollback to previous version
- `validate_row_count()` — assertion-style row count check
- `get_previous_version()` — find the previous version of a Delta table for rollback

In [0]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col


def log_execution(catalog_name, metadata_schema, table_name, city, status,
                  source_row_count, target_row_count, file_location):
    """Append one row to metadata.execution_log.

    Called at the end of every pipeline step (Raw, Bronze, Silver, Gold).
    Always called — both on SUCCESS and on FAILED — so failed runs are auditable.
    """
    log_row = spark.createDataFrame([Row(
        table_name       = table_name,
        city             = city,
        execution_time   = datetime.now(),
        status           = status,
        source_row_count = int(source_row_count),
        target_row_count = int(target_row_count),
        file_location    = file_location,
        created_date     = datetime.now().date()
    )])
    (log_row.write.format("delta").mode("append")
        .saveAsTable(f"{catalog_name}.{metadata_schema}.execution_log"))
    print(f"  logged to execution_log: {table_name} | {status} | src={source_row_count} tgt={target_row_count}")


def log_dqx(catalog_name, metadata_schema, table_name, city,
            total_records, passed_records, failed_records):
    """Append one row to metadata.dqx_execution_log.

    Called once per Silver transformation with pass/fail counts from DQX expectations.
    """
    log_row = spark.createDataFrame([Row(
        table_name      = table_name,
        city            = city,
        execution_time  = datetime.now(),
        total_records   = int(total_records),
        passed_records  = int(passed_records),
        failed_records  = int(failed_records),
        created_date    = datetime.now().date()
    )])
    (log_row.write.format("delta").mode("append")
        .saveAsTable(f"{catalog_name}.{metadata_schema}.dqx_execution_log"))
    pct = round(100 * passed_records / total_records, 2) if total_records else 0.0
    print(f"  logged to dqx_execution_log: {table_name} | {passed_records}/{total_records} passed ({pct}%)")


def get_previous_version(full_table_name):
    """Return the version number immediately before the current one.

    Used for rollback on failure. Returns None if the table has no prior version
    (i.e., this was the first write).
    """
    try:
        history = spark.sql(f"DESCRIBE HISTORY {full_table_name}").orderBy(col("version").desc()).collect()
        if len(history) < 2:
            return None
        return history[1]["version"]
    except Exception:
        return None


def rollback_table(full_table_name):
    """Roll a Delta table back to its previous version (not VERSION 0).

    Returns True if rollback succeeded, False if there was no prior version.
    """
    prev = get_previous_version(full_table_name)
    if prev is None:
        print(f"   cannot rollback {full_table_name}: no prior version")
        return False
    spark.sql(f"RESTORE TABLE {full_table_name} TO VERSION AS OF {prev}")
    print(f"   rolled back {full_table_name} to version {prev}")
    return True


def validate_row_count(source_count, target_count, label=""):
    """Return 'SUCCESS' if counts match, 'FAILED' otherwise. Prints result."""
    if source_count == target_count:
        print(f"  row count validated{' for '+label if label else ''}: {source_count}")
        return "SUCCESS"
    else:
        print(f"  row count mismatch{' for '+label if label else ''}: source={source_count} target={target_count}")
        return "FAILED"


print("_common_helpers loaded: log_execution, log_dqx, rollback_table, get_previous_version, validate_row_count")